In [0]:
dbutils.widgets.text("catalog_name", "workspace", "Catalog Name")
dbutils.widgets.text("schema_prefix", "retail", "Schema Prefix")

In [0]:
catalog = dbutils.widgets.get("catalog_name")
schema_prefix = dbutils.widgets.get("schema_prefix")

gold_db = f"{catalog}.{schema_prefix}_gold"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE {gold_db}")

print("Optimizing tables in:", gold_db)

Optimizing tables in: workspace.retail_gold


In [0]:
print("Optimizing FactSales...")
spark.sql(f"OPTIMIZE {gold_db}.FactSales ZORDER BY (order_purchase_date_key, product_key)")
print("Done.")

Optimizing FactSales...
Done.


In [0]:
display(spark.sql(f"DESCRIBE HISTORY {gold_db}.FactSales").select("version", "timestamp", "operation"))

version,timestamp,operation
0,2026-08-08T06:55:14.000Z,CREATE OR REPLACE TABLE AS SELECT


In [0]:
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table(f"{gold_db}.FactSales")
print(f"Version 0 row count: {df_v0.count()}")
print(f"Current row count: {spark.table(f'{gold_db}.FactSales').count()}")

Version 0 row count: 112650
Current row count: 112650


In [0]:
result = spark.sql(f"VACUUM {gold_db}.FactSales")
display(result)
print("Vacuum complete (default 7-day retention kept).")

path
""


Vacuum complete (default 7-day retention kept).
